# 02. Data Cleaning & Preprocessing
Runs the full data pipeline: raw → cleaned → processed CSV.
Uses `PathConfig` for dynamic path resolution across Windows and Google Colab.

In [ ]:
import sys, os

# --- Colab / Windows path bootstrap ---
if 'google.colab' in sys.modules:
    %cd /content/house-pridiction
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    os.chdir(project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.config import PathConfig, ModelConfig

print(f"Project root    : {os.getcwd()}")
print(f"Raw data path   : {PathConfig.RAW_DATA}")
print(f"Processed path  : {PathConfig.PROCESSED_DATA}")

## 1. Load Raw Data

In [ ]:
if not os.path.exists(PathConfig.RAW_DATA):
    print(f"❌ Raw data not found: {PathConfig.RAW_DATA}")
else:
    df_raw = pd.read_csv(PathConfig.RAW_DATA)
    print(f"✅ Raw data loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
    df_raw.head(3)

## 2. Run the Full Cleaning & Feature Engineering Pipeline
This calls the standardized `make_dataset` module which:
- Drops high-missing columns
- Handles missing values
- Handles outliers
- Engineers features
- Saves `housing_cleaned.csv`

In [ ]:
# Run make_dataset directly (same as: python -m src.data.make_dataset)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'src.data.make_dataset'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"⚠️ STDERR:\n{result.stderr}")
else:
    print("✅ make_dataset completed successfully.")

## 3. Inspect Processed Output

In [ ]:
if not os.path.exists(PathConfig.PROCESSED_DATA):
    print(f"❌ Processed file not found: {PathConfig.PROCESSED_DATA}")
else:
    df_clean = pd.read_csv(PathConfig.PROCESSED_DATA)
    print(f"✅ Processed data loaded: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
    print(f"\nMissing values: {df_clean.isnull().sum().sum()}")
    print(f"\nTarget column '{ModelConfig.TARGET_COL}' stats (log-scale):")
    print(df_clean[ModelConfig.TARGET_COL].describe().round(4))
    print(f"\nApprox. dollar range: ${np.expm1(df_clean[ModelConfig.TARGET_COL].min()):,.0f} - ${np.expm1(df_clean[ModelConfig.TARGET_COL].max()):,.0f}")

## 4. Feature Distribution Summary

In [ ]:
numeric_cols = df_clean.select_dtypes(include='number').columns.tolist()
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f"Numeric features : {len(numeric_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"\nCategorical columns: {cat_cols}")